# 1. MLP and Input-Output Convex Neural Networks

# 1.2 MLP implementation from scratch

In [1]:
%pip install numpy datasets matplotlib wandb

  Using cached pyyaml-6.0.3-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 55.2 kB/s  0:04:51 eta 0:00:040:00:09
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 40.1 kB/s  0:00:15 37.3 kB/s eta 0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.9/839.9 kB 47.9 kB/s  0:00:167.7 kB/s eta 0:00:02:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 53.1 kB/s  0:01:20 eta 0:00:010:06:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 60.7 kB/s  0:02:47a 0:00:030:00:07:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 57.6 kB/s  0:05:56 eta 0:00:030:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 40.9 kB/s  0:00:48 40.2 kB/s eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 47.8 kB/s  0:00:396.7 kB/s eta 0:00:02:08
Using cached cycler-0.12.1-py3-none-any

In [1]:
import numpy as np

ROLL_NUMBER = 2026701036
SEED = ROLL_NUMBER % 100000
RUN = 0
print("Seed for question 1:", SEED)

Seed for question 1: 1036


In [2]:
from datasets import load_dataset

ds = load_dataset("uoft-cs/cifar10")

X = np.empty((60_000,32,32,3),dtype=np.uint8)
y = np.empty(60_000, dtype=np.int64)

for i, img in enumerate(ds["train"]["img"]):
    X[i] = np.asarray(img, dtype=np.uint8)
for i, img in enumerate(ds["test"]["img"]):
    X[50_000 + i] = np.asarray(img, dtype=np.uint8)

y[:50_000] = np.asarray(ds["train"]["label"], dtype=np.int64)
y[50_000:] = np.asarray(ds["test"]["label"], dtype=np.int64)

print("X:", X.shape,X.dtype)
print("y:", y.shape,y.dtype)
print("per-class counts:", np.bincount(y, minlength=10))

/home/surya/work/research/SMAI-A4/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X: (60000, 32, 32, 3) uint8
y: (60000,) int64
per-class counts: [6000 6000 6000 6000 6000 6000 6000 6000 6000 6000]


In [3]:
X = X.reshape(60_000,32*32*3)

def stratified_split(y, rng, n_train, n_val, n_test):
    train_idx, val_idx, test_idx = [], [], []
    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        idx = rng.permutation(idx)
        assert len(idx) >= n_train+n_val+n_test
        train_idx.append(idx[:n_train])
        val_idx.append(idx[n_train:n_train+n_val])
        test_idx.append(idx[n_train+n_val:n_train+n_val+n_test])
    return (np.concatenate(train_idx),np.concatenate(val_idx),np.concatenate(test_idx))

rng_split = np.random.default_rng(SEED+100*RUN+0)
train_idx,val_idx,test_idx = stratified_split(y,rng_split,3600,1200,1200)

print("train counts:", np.bincount(y[train_idx],minlength=10))
print("val counts:", np.bincount(y[val_idx],minlength=10))
print("test counts:", np.bincount(y[test_idx],minlength=10))

train counts: [3600 3600 3600 3600 3600 3600 3600 3600 3600 3600]
val counts: [1200 1200 1200 1200 1200 1200 1200 1200 1200 1200]
test counts: [1200 1200 1200 1200 1200 1200 1200 1200 1200 1200]


In [4]:
X_train = X[train_idx].astype(np.float64)
X_val = X[val_idx].astype(np.float64)
X_test = X[test_idx].astype(np.float64)

X_train /= 255.0
X_val /= 255.0
X_test /= 255.0

y_train = y[train_idx].copy()
y_val = y[val_idx].copy()
y_test = y[test_idx].copy()

train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

train_std[train_std ==0.0] = 1.0

X_train -= train_mean
X_train /= train_std
X_val -= train_mean
X_val /= train_std
X_test -= train_mean
X_test /= train_std

print("train mean first 3:", X_train.mean(axis=0)[:3])
print("train std first 3:", X_train.std(axis=0)[:3])
print("test mean first 3:", X_test.mean(axis=0)[:3])
print("test std first 3:", X_test.std(axis=0)[:3])

print("X_train:", X_train.shape, X_train.dtype)
print("X_val:", X_val.shape, X_val.dtype)
print("X_test:", X_test.shape, X_test.dtype)

train mean first 3: [1.40387597e-14 4.61716899e-14 4.00795570e-14]
train std first 3: [1. 1. 1.]
test mean first 3: [ 0.00231415  0.00064378 -0.00472698]
test std first 3: [1.00449637 1.00379161 1.00436622]
X_train: (36000, 3072) float64
X_val: (12000, 3072) float64
X_test: (12000, 3072) float64


the test mean and std of the above results are not exactly 0 and 1 because the standardisation statistics were computed on the training split only, the test split is different sample and therefore has slightly differnet statis

In [5]:
print(y_train.dtype, y_train.shape, np.unique(y_train))
print(y_val.dtype, y_val.shape, np.unique(y_val))
print(y_test.dtype, y_test.shape, np.unique(y_test))

int64 (36000,) [0 1 2 3 4 5 6 7 8 9]
int64 (12000,) [0 1 2 3 4 5 6 7 8 9]
int64 (12000,) [0 1 2 3 4 5 6 7 8 9]


In [6]:
class Linear:
    def __init__(self,n_in,n_out,activation,rng):
        self.n_in = n_in
        self.n_out = n_out
        self.activation = activation

        # for relu variance 2/n_in and for all other 1/n_in
        variance = 2.0/n_in if isinstance(activation,ReLU) else 1.0/n_in

        self.W = rng.normal(0.0,np.sqrt(variance), size=(n_in,n_out))
        self.b = np.zeros(n_out, dtype=np.float64)

        #cumulative gradients summer over the train call
        self.dW = np.zeros_like(self.W)
        self.db = np.zeros_like(self.b)

        # forward pass cache
        self.x = None #input to the layer
        self.z = None #pre-activation
        self.h = None #post-activation output

    def forward(self, x):
        self.x = x
        self.z = x @ self.W + self.b
        self.h = self.activation.forward(self.z)
        return self.h

    def backward(self, dout):
        # gradient of output dout = dL/dh for thhis layers output
        dz = dout * self.activation.backward(self.z)
        self.dW += self.x.T@dz
        self.db += dz.sum(axis=0)
        return dz @ self.W.T

    def zero_grad(self):
        self.dW.fill(0.0)
        self.db.fill(0.0)

In [7]:
class ReLU:
    def forward(self, z):
        return np.maximum(0.0,z)
    def backward(self, z):
        return (z > 0.0).astype(np.float64)

class Tanh:
    def forward(self, z):
        return np.tanh(z)
    def backward(self, z):
        t = np.tanh(z)
        return 1.0 - t*t

class Sigmoid:
    def forward(self, z):
        z = np.asarray(z, dtype=np.float64)
        e = np.exp(-np.abs(z))
        return np.where(z >= 0.0,1.0/(1.0+e),e/(1.0+e))
    def backward(self, z):
        s = self.forward(z)
        return s*(1.0-s)

class Identity:
    def forward(self, z):
        return z
    def backward(self, z):
        return np.ones_like(z, dtype=np.float64)

In [8]:
# quick test for the above classes implemented

rng_test = np.random.default_rng(SEED + 100 * RUN + 1)

layer = Linear(3,4,ReLU(),rng_test)
x = np.random.default_rng(123).normal(size=(5,3))
h = layer.forward(x)

print("W:", layer.W.shape, "b:", layer.b.shape)
print("z:", layer.z.shape, "h:", h.shape)
print("bias is zero:", np.all(layer.b == 0.0))
print("grads start at zero:", np.all(layer.dW == 0.0), np.all(layer.db == 0.0))

dx = layer.backward(np.ones_like(h))
print("dx:", dx.shape)
print("dW:", layer.dW.shape, "db:", layer.db.shape)

layer.zero_grad()
print("after zero_grad:", np.all(layer.dW == 0.0), np.all(layer.db == 0.0))

rng_v = np.random.default_rng(7)
relu_layer = Linear(1000, 10, ReLU(), rng_v)
tanh_layer = Linear(1000, 10, Tanh(), rng_v)

print("ReLU W variance:", relu_layer.W.var(), "expected:", 2/1000)
print("Tanh W variance:", tanh_layer.W.var(), "expected:", 1/1000)

W: (3, 4) b: (4,)
z: (5, 4) h: (5, 4)
bias is zero: True
grads start at zero: True True
dx: (5, 3)
dW: (3, 4) db: (4,)
after zero_grad: True True
ReLU W variance: 0.0019771671435858808 expected: 0.002
Tanh W variance: 0.0009811740048855106 expected: 0.001


In [12]:
# model classes and losses

class Model:
    def __init__(self, layers, loss_type):
        self.layers = layers
        self.loss_type = loss_type.lower()

        if self.loss_type not in ("mse","bce","cce"):
            raise ValueError("loss_type must be either mse or bce or cce")
        self._accum_count = 0

    def forward(self, x):
        h = x
        for layer in self.layers:
            h = layer.forward(h)
        return h
    
    def backward(self, dout):
        g = dout
        for layer in reversed(self.layers):
            g = layer.backward(g)
        return g
    
    def zero_grad(self):
        for layer in self.layers:
            layer.zero_grad()
        self._accum_count = 0

    def train(self, x, y):
        out = self.forward(x)
        loss, dout = self._loss(out,y)
        self.backward(dout)
        self._accum_count += 1
        return loss

    def update(self, lr):
        m = max(1, self._accum_count)
        for layer in self.layers:
            layer.W -= lr * layer.dW/m
            layer.b -= lr * layer.db/m
        self.zero_grad()

    def predict(self, x):
        out = self.forward(x)
        if self.loss_type == "cce":
            return self._softmax(out)
        return out

    def _softmax(self, logits):
        m = logits.max(axis=1, keepdims=True)
        e = np.exp(logits - m)
        return e/e.sum(axis=1, keepdims=True)

    def _loss(self, out, y):
        if self.loss_type == "mse":
            target = np.asarray(y, dtype=np.float64)
            if target.shape != out.shape:
                target = target.reshape(out.shape)
            diff = out - target
            loss = np.mean(diff*diff)
            dout = 2.0 * diff/diff.size
            return loss, dout

        if self.loss_type == "bce":
            p = np.clip(out, 1e-12, 1.0 - 1e-12)
            target = np.asarray(y, dtype=np.float64).reshape(p.shape)
            loss = -np.mean(target*np.log(p) + (1.0-target)*np.log(1.0-p))
            dout = (-target/p + (1.0 - target) / (1.0 -p))/p.size
            return loss, dout

        logits = np.asarray(out, dtype=np.float64)
        labels = np.asarray(y, dtype=np.int64).reshape(-1)
        B = logits.shape[0]

        m = logits.max(axis=1, keepdims=True)
        exp = np.exp(logits - m)
        sumexp = exp.sum(axis=1, keepdims=True)
        p = exp/sumexp

        logsumexp = (m + np.log(sumexp))[:,0]
        loss = np.mean(logsumexp - logits[np.arange(B), labels])

        dout = p
        dout[np.arange(B), labels] -= 1.0
        dout /= B

        return loss, dout

    def save_to(self, path):
        path = str(path)
        if not path.lower().endswith(".npz"):
            path += ".npz"

        arrays = {}
        for i, layer in enumerate(self.layers):
            arrays[f"W{i}"] = layer.W
            arrays[f"b{i}"] = layer.b

        np.savez(path, **arrays)

    def load_from(self, path):
        path = str(path)
        if not path.lower().endswith(".npz"):
            path += ".npz"
        with np.load(path) as data:
            if len(data.files) != 2*len(self.layers):
                raise ValueError("wrong no of parameter array")
            for i, layer in enumerate(self.layers):
                try:
                    W = data[f"W{i}"]
                    b = data[f"b{i}"]
                except KeyError as exc:
                    raise ValueError(f"mmissing parameters for layer {i}") from exc
                if W.shape != layer.W.shape or b.shape != layer.b.shape:
                    raise ValueError(f"mismath in layer {i}: saved {W.shape}/{b.shape}, current {layer.W.shape}/{layer.b.shape}")

            for i, layer in enumerate(self.layers):
                layer.W[...] = data[f"W{i}"]
                layer.b[...] = data[f"b{i}"]
        self.zero_grad()

In [13]:
#test cell
rng_cce = np.random.default_rng(SEED + 100 * RUN + 1)
cce_model = Model(
    [Linear(2, 5, Tanh(), rng_cce), Linear(5, 3, Identity(), rng_cce)],
    "cce",
)
xb = np.random.default_rng(1).normal(size=(4, 2))
yb = np.array([0, 1, 2, 1])
loss = cce_model.train(xb, yb)
probs = cce_model.predict(xb)

print("CCE loss:", loss)
print("probs shape:", probs.shape)
print("row sums:", probs.sum(axis=1))
print("grads nonzero:", [np.any(l.dW != 0.0) for l in cce_model.layers])

W_before = cce_model.layers[0].W.copy()
cce_model.update(0.1)
print("weights changed:", not np.allclose(W_before, cce_model.layers[0].W))
print("grads zero after update:", [np.all(l.dW == 0.0) for l in cce_model.layers])

rng_bce = np.random.default_rng(SEED + 100 * RUN + 1)
bce_model = Model(
    [Linear(2, 4, Tanh(), rng_bce), Linear(4, 1, Sigmoid(), rng_bce)],
    "bce",
)
xb = np.random.default_rng(2).normal(size=(6, 2))
yb = np.array([0, 1, 1, 0, 1, 0])
loss = bce_model.train(xb, yb)
p = bce_model.predict(xb)
print("BCE loss:", loss)
print("p shape:", p.shape, "p range:", p.min(), p.max())

rng_mse = np.random.default_rng(SEED + 100 * RUN + 1)
mse_model = Model(
    [Linear(2, 3, Tanh(), rng_mse), Linear(3, 2, Identity(), rng_mse)],
    "mse",
)
xb = np.random.default_rng(3).normal(size=(5, 2))
yb = np.random.default_rng(4).normal(size=(5, 2))
loss = mse_model.train(xb, yb)
print("MSE loss:", loss, "pred shape:", mse_model.predict(xb).shape)

cce_model.save_to("/tmp/cce_model.npz")
rng_copy = np.random.default_rng(123)
cce_copy = Model(
    [Linear(2, 5, Tanh(), rng_copy), Linear(5, 3, Identity(), rng_copy)],
    "cce",
)
cce_copy.load_from("/tmp/cce_model.npz")
print("load matches:", all(
    np.allclose(a.W, b.W) and np.allclose(a.b, b.b)
    for a, b in zip(cce_model.layers, cce_copy.layers)
))

try:
    rng_bad = np.random.default_rng(456)
    bad_model = Model(
        [Linear(2, 4, Tanh(), rng_bad), Linear(4, 3, Identity(), rng_bad)],
        "cce",
    )
    bad_model.load_from("/tmp/cce_model.npz")
except ValueError as exc:
    print("expected architecture error:", exc)

logits = np.array([
    [1000.0, 1001.0, 1002.0],
    [-1000.0, -1001.0, -1002.0],
])
labels = np.array([2, 0])
loss, grad = cce_model._loss(logits, labels)

print("large-logit CCE loss:", loss)
print("finite:", np.isfinite(loss))
print("grad row sums:", grad.sum(axis=1))

CCE loss: 1.3393735785610952
probs shape: (4, 3)
row sums: [1. 1. 1. 1.]
grads nonzero: [np.True_, np.True_]
weights changed: True
grads zero after update: [np.True_, np.True_]
BCE loss: 0.770619118340535
p shape: (6, 1) p range: 0.3173189895670321 0.5992580508308385
MSE loss: 2.5521364319456263 pred shape: (5, 2)
load matches: True
expected architecture error: mismath in layer 0: saved (2, 5)/(5,), current (2, 4)/(4,)
large-logit CCE loss: 0.40760596444442854
finite: True
grad row sums: [-5.55111512e-17 -5.55111512e-17]


In [ ]:
def evaluate(model, X, y, batch_size=1024):
    n = len(X)
    total_loss = 0.0
    total_count